In [1]:
"""
SENTINEL 8.1 — Quant Engine (Price/Trend only; IA se integra después)
Autor: Usted + ChatGPT (versión integrada)

FOCO:
- Optimización: Sharpe NETO (costos dentro del retorno), con robustez:
  - Purged Walk-Forward CV + Embargo correcto
  - Score robusto: avg Sharpe - penalización por peor DD (cap) - penalización por variabilidad entre folds
- Motor de cartera explícito:
  - Pesos equiponderados SOLO entre posiciones activas (pos=1)
  - Rebalanceo por EVENTO (recomendado) o CALENDARIO (opcional)
  - Costos por trading vía turnover y financiamiento vía exposición
- Sin leakage:
  - PROHIBIDO bfill
  - ffill limitado solo para huecos cortos (limit=5)
- Benchmarks y atribución:
  - Alfa/Beta por OLS con intercepto y errores Newey–West
- Stress y robustez:
  - Stress por episodios reales
  - Bootstrap por bloques (DD y prob. de ruina)
- Reportes y gráficas:
  - Equity vs SPY/QQQ
  - Drawdown
  - Rolling Sharpe / Rolling Vol
  - Distribución retornos
  - Exposición y Turnover
  - Top params table (CV)

Requisitos:
pip install yfinance pandas numpy matplotlib
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional


# =============================================================================
# CONFIGS
# =============================================================================

@dataclass
class CostsConfig:
    commission: float = 0.001      # 0.1% por trade (aprox) aplicado vía turnover
    swap_daily: float = 0.0003     # 0.03% diario (si aplica, p.ej. CFD)
    slippage: float = 0.0002       # 0.02% por trade (proxy) aplicado vía turnover
    apply_swap: bool = True
    apply_slippage: bool = True


@dataclass
class BacktestConfig:
    capital_initial: float = 100000.0
    start: str = "2006-01-01"
    end: str = "2026-02-20"
    burn_in: int = 200
    trading_days: int = 252
    rf_annual: float = 0.0
    # Cartera:
    rebalance_mode: str = "event"  # "event" (recomendado) o "calendar"
    rebalance_freq: str = "W-FRI"  # si calendar: "W-FRI" o "M"


@dataclass
class CVConfig:
    train_years: int = 8
    test_years: int = 1
    purge_days: int = 10
    embargo_days: int = 5


@dataclass
class ScoreConfig:
    dd_cap: float = -0.35      # CAP DD (ej. -35%)
    lambda_dd: float = 2.0     # penalización por exceder dd_cap (en exceso absoluto)
    lambda_var: float = 0.5    # penalización por variabilidad entre folds (std sharpe)
    min_splits: int = 5


# Escenarios (puede ampliar)
ESCENARIOS: Dict[str, List[str]] = {
    "tech_original": ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'AVGO', 'ASML', 'TSM', 'ADBE', 'NFLX', 'AMD'],
    "tech_alternativo": ['CRM', 'ORCL', 'IBM', 'INTC', 'CSCO', 'QCOM', 'TXN', 'ADI', 'MU', 'ANET', 'PLTR', 'SNPS'],
    "defensivo": ['JNJ', 'PG', 'KO', 'PEP', 'WMT', 'COST', 'MO', 'CL', 'GIS', 'KMB', 'MDLZ', 'CPB'],
    "industrial": ['HON', 'CAT', 'DE', 'BA', 'LMT', 'GE', 'MMM', 'EMR', 'ITW', 'ETN', 'ROK', 'DOV'],
    "aleatorio": ['AAPL', 'MSFT', 'JNJ', 'PG', 'CAT', 'XOM', 'CVX', 'JPM', 'GS', 'V', 'MA', 'KO'],
}

# Episodios stress (realistas; ajuste si desea)
STRESS_EPISODES = {
    "GFC_2008": ("2008-01-01", "2009-06-30"),
    "EURO_2011": ("2011-07-01", "2011-12-31"),
    "Q4_2018": ("2018-10-01", "2018-12-31"),
    "COVID_2020": ("2020-02-15", "2020-06-30"),
    "RATES_2022": ("2022-01-01", "2022-12-31"),
}


# =============================================================================
# INDICATOR: KAMA
# =============================================================================

def kama(price: np.ndarray, n: int = 10, fast: int = 2, slow: int = 30) -> np.ndarray:
    price = np.asarray(price, dtype=float)
    length = len(price)
    out = np.full(length, np.nan)
    if length == 0:
        return out

    out[0] = price[0]
    fast_sc = 2.0 / (fast + 1.0)
    slow_sc = 2.0 / (slow + 1.0)

    for i in range(1, length):
        if i < n:
            out[i] = price[i]
        else:
            change = abs(price[i] - price[i - n])
            volatility = np.sum(np.abs(np.diff(price[i - n + 1:i + 1])))
            er = 0.0 if volatility == 0 else change / volatility
            sc = (er * (fast_sc - slow_sc) + slow_sc) ** 2
            out[i] = out[i - 1] + sc * (price[i] - out[i - 1])
    return out


# =============================================================================
# DATA: DOWNLOAD + ALIGN (NO bfill)
# =============================================================================

def download_prices(tickers: List[str], start: str, end: str) -> pd.DataFrame:
    """
    Descarga precios ajustados y alinea a calendario hábil.
    - NO bfill (evita leakage).
    - ffill limitado para huecos cortos (limit=5).
    """
    raw = yf.download(
        tickers, start=start, end=end, auto_adjust=True,
        group_by="ticker", progress=False
    )

    idx = pd.bdate_range(start=start, end=end)
    df = pd.DataFrame(index=idx)

    for t in tickers:
        try:
            if isinstance(raw, pd.DataFrame) and "Close" in raw.columns:
                s = raw["Close"]
                if isinstance(s, pd.DataFrame):
                    s = s[t] if t in s.columns else None
            else:
                s = raw[t]["Close"]

            if s is None:
                continue

            s = pd.Series(s).dropna()
            if s.empty:
                continue

            s = s.reindex(idx)
            s = s.ffill(limit=5)
            df[t] = s
        except Exception:
            continue

    return df.dropna(how="all")


# =============================================================================
# POSITIONS & WEIGHTS
# =============================================================================

def compute_positions_kama(prices: pd.DataFrame, params: Dict, burn_in: int) -> pd.DataFrame:
    n, fast, slow = params["n"], params["fast"], params["slow"]
    pos = pd.DataFrame(index=prices.index, columns=prices.columns, dtype=float)

    for t in prices.columns:
        p = prices[t]
        if p.isna().all():
            pos[t] = 0.0
            continue

        k = pd.Series(kama(p.values, n=n, fast=fast, slow=slow), index=p.index).shift(1)
        sig = ((p > k) & p.notna() & k.notna()).astype(float)
        sig.iloc[:burn_in] = 0.0
        pos[t] = sig.fillna(0.0)

    return pos.fillna(0.0)


def event_rebalance_weights(pos: pd.DataFrame) -> pd.DataFrame:
    active = (pos > 0).astype(float)
    n_active = active.sum(axis=1).replace(0, np.nan)
    w = active.div(n_active, axis=0).fillna(0.0)
    return w


def calendar_rebalance_weights(pos: pd.DataFrame, freq: str = "W-FRI") -> pd.DataFrame:
    w_event = event_rebalance_weights(pos)
    reb_dates = w_event.resample(freq).last().index
    w = pd.DataFrame(index=w_event.index, columns=w_event.columns, dtype=float)

    last_w = np.zeros(len(w_event.columns))
    for dt in w_event.index:
        if dt in reb_dates:
            last_w = w_event.loc[dt].values
        w.loc[dt] = last_w

    return w.fillna(0.0)


def compute_turnover(weights: pd.DataFrame) -> pd.Series:
    dw = weights.diff().abs().fillna(0.0)
    return 0.5 * dw.sum(axis=1)


def apply_trading_costs(port_ret: pd.Series, weights: pd.DataFrame, costs: CostsConfig) -> pd.Series:
    to = compute_turnover(weights)
    trade_cost = to * (costs.commission + (costs.slippage if costs.apply_slippage else 0.0))

    exposure = weights.abs().sum(axis=1)
    swap_cost = exposure * (costs.swap_daily if costs.apply_swap else 0.0)

    net = port_ret - trade_cost - swap_cost
    return net


def backtest_portfolio(prices: pd.DataFrame, params: Dict, bt_cfg: BacktestConfig, costs: CostsConfig) -> Dict:
    pos = compute_positions_kama(prices, params, bt_cfg.burn_in)

    if bt_cfg.rebalance_mode == "calendar":
        w = calendar_rebalance_weights(pos, freq=bt_cfg.rebalance_freq)
    else:
        w = event_rebalance_weights(pos)

    rets = prices.pct_change()
    port_gross = (w.shift(1).fillna(0.0) * rets.fillna(0.0)).sum(axis=1)
    port_net = apply_trading_costs(port_gross, w, costs)
    port_net = port_net.replace([np.inf, -np.inf], 0.0).fillna(0.0)

    equity = bt_cfg.capital_initial * (1.0 + port_net).cumprod()

    return {
        "returns_gross": port_gross,
        "returns_net": port_net,
        "equity": equity,
        "positions": pos,
        "weights": w,
        "turnover": compute_turnover(w),
    }


# =============================================================================
# METRICS
# =============================================================================

def max_drawdown(equity: pd.Series) -> float:
    peak = equity.cummax()
    dd = equity / peak - 1.0
    return float(dd.min())


def annualize_return(ret: pd.Series, trading_days: int = 252) -> float:
    if len(ret) == 0:
        return 0.0
    total = (1.0 + ret).prod() - 1.0
    return float((1.0 + total) ** (trading_days / len(ret)) - 1.0)


def sharpe_ratio(ret: pd.Series, rf_annual: float = 0.0, trading_days: int = 252) -> float:
    rf_daily = (1.0 + rf_annual) ** (1.0 / trading_days) - 1.0
    ex = ret - rf_daily
    sd = ex.std(ddof=1)
    if sd == 0 or np.isnan(sd):
        return 0.0
    return float(np.sqrt(trading_days) * ex.mean() / sd)


def sortino_ratio(ret: pd.Series, rf_annual: float = 0.0, trading_days: int = 252) -> float:
    rf_daily = (1.0 + rf_annual) ** (1.0 / trading_days) - 1.0
    ex = ret - rf_daily
    downside = ex.copy()
    downside[downside > 0] = 0
    dd = downside.std(ddof=1)
    if dd == 0 or np.isnan(dd):
        return 0.0
    return float(np.sqrt(trading_days) * ex.mean() / dd)


def cvar(ret: pd.Series, alpha: float = 0.05) -> float:
    r = ret.dropna().values
    if len(r) == 0:
        return np.nan
    q = np.quantile(r, alpha)
    tail = r[r <= q]
    return float(tail.mean()) if len(tail) else float(q)


def summarize_performance(bt: Dict, bt_cfg: BacktestConfig) -> Dict:
    r = bt["returns_net"]
    eq = bt["equity"]

    ann = annualize_return(r, bt_cfg.trading_days)
    sr = sharpe_ratio(r, bt_cfg.rf_annual, bt_cfg.trading_days)
    so = sortino_ratio(r, bt_cfg.rf_annual, bt_cfg.trading_days)
    mdd = max_drawdown(eq)  # negativo
    calmar = ann / abs(mdd) if mdd != 0 else np.inf

    to_ann = float(bt["turnover"].sum() * (bt_cfg.trading_days / len(r))) if len(r) else 0.0
    exp_avg = float(bt["weights"].abs().sum(axis=1).mean())

    return {
        "ann_return": ann,
        "sharpe": sr,
        "sortino": so,
        "max_dd": mdd,
        "calmar": calmar,
        "turnover_annualized": to_ann,
        "avg_exposure": exp_avg,
        "cvar_5": cvar(r, 0.05),
        "cvar_1": cvar(r, 0.01),
    }


# =============================================================================
# ALPHA/BETA OLS + NEWEY-WEST
# =============================================================================

def newey_west_se(x: np.ndarray, resid: np.ndarray, lags: int = 5) -> np.ndarray:
    T, k = x.shape
    S = np.zeros((k, k))

    for t in range(T):
        xt = x[t:t+1].T
        S += (resid[t] ** 2) * (xt @ xt.T)

    for L in range(1, lags + 1):
        w = 1.0 - L / (lags + 1.0)
        Gamma = np.zeros((k, k))
        for t in range(L, T):
            xt = x[t:t+1].T
            xtL = x[t-L:t-L+1].T
            Gamma += resid[t] * resid[t-L] * (xt @ xtL.T)
        S += w * (Gamma + Gamma.T)

    XtX_inv = np.linalg.inv(x.T @ x)
    V = XtX_inv @ S @ XtX_inv
    return np.sqrt(np.diag(V))


def alpha_beta_nw(ret_strategy: pd.Series, ret_mkt: pd.Series,
                  rf_annual: float = 0.0, trading_days: int = 252, lags: int = 5) -> Dict:
    rf_daily = (1.0 + rf_annual) ** (1.0 / trading_days) - 1.0
    df = pd.DataFrame({"s": ret_strategy, "m": ret_mkt}).dropna()
    if len(df) < 50:
        return {"alpha_annual": np.nan, "beta": np.nan, "t_alpha": np.nan}

    y = (df["s"] - rf_daily).values
    x1 = (df["m"] - rf_daily).values
    X = np.column_stack([np.ones(len(df)), x1])

    beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta_hat

    se = newey_west_se(X, resid, lags=lags)
    t_alpha = beta_hat[0] / se[0] if se[0] != 0 else np.nan

    alpha_daily = beta_hat[0]
    alpha_annual = (1.0 + alpha_daily) ** trading_days - 1.0

    return {"alpha_annual": float(alpha_annual), "beta": float(beta_hat[1]), "t_alpha": float(t_alpha)}


# =============================================================================
# PURGED WALK-FORWARD + EMBARGO (CORRECT)
# =============================================================================

def purged_walk_forward_splits(index: pd.DatetimeIndex, cv_cfg: CVConfig) -> List[Tuple[pd.DatetimeIndex, pd.DatetimeIndex]]:
    years = sorted(index.year.unique())
    splits = []

    for i in range(cv_cfg.train_years, len(years) - cv_cfg.test_years + 1):
        train_years = years[i - cv_cfg.train_years:i]
        test_years = years[i:i + cv_cfg.test_years]

        train_idx = index[index.year.isin(train_years)]
        test_idx = index[index.year.isin(test_years)]
        if len(train_idx) == 0 or len(test_idx) == 0:
            continue

        test_start_real = test_idx.min()

        # Purge train end
        if len(train_idx) > cv_cfg.purge_days:
            train_idx = train_idx[:-cv_cfg.purge_days]

        # Purge test start
        if len(test_idx) > cv_cfg.purge_days:
            test_idx = test_idx[cv_cfg.purge_days:]

        if len(train_idx) == 0 or len(test_idx) == 0:
            continue

        # Embargo: train must end <= test_start_real - embargo_days
        embargo_cut = test_start_real - pd.tseries.offsets.BDay(cv_cfg.embargo_days)
        train_idx = train_idx[train_idx <= embargo_cut]

        if len(train_idx) < 200 or len(test_idx) < 100:
            continue

        splits.append((train_idx, test_idx))

    return splits


# =============================================================================
# ROBUST SCORE (AVG SHARPE - DD CAP PENALTY - VARIANCE PENALTY)
# =============================================================================

def score_params_cv(prices: pd.DataFrame, params: Dict, bt_cfg: BacktestConfig,
                    costs: CostsConfig, cv_cfg: CVConfig, sc_cfg: ScoreConfig) -> Dict:
    splits = purged_walk_forward_splits(prices.index, cv_cfg)
    if len(splits) < sc_cfg.min_splits:
        return {"score": -np.inf, "avg_sharpe": np.nan, "std_sharpe": np.nan,
                "worst_dd": np.nan, "avg_calmar": np.nan, "splits": len(splits)}

    sharpes, calmars, maxdds = [], [], []
    for _, test_idx in splits:
        bt = backtest_portfolio(prices.loc[test_idx], params, bt_cfg, costs)
        m = summarize_performance(bt, bt_cfg)
        sharpes.append(m["sharpe"])
        calmars.append(m["calmar"])
        maxdds.append(m["max_dd"])

    avg_sh = float(np.nanmean(sharpes))
    std_sh = float(np.nanstd(sharpes))
    avg_ca = float(np.nanmean(calmars))
    worst_dd = float(np.nanmin(maxdds))  # más negativo

    # exceso sobre el cap
    dd_excess = max(0.0, abs(worst_dd) - abs(sc_cfg.dd_cap))
    score = avg_sh - sc_cfg.lambda_dd * dd_excess - sc_cfg.lambda_var * std_sh

    return {
        "score": float(score),
        "avg_sharpe": avg_sh,
        "std_sharpe": std_sh,
        "avg_calmar": avg_ca,
        "worst_dd": worst_dd,
        "splits": len(splits),
    }


def grid_search_kama(prices: pd.DataFrame, bt_cfg: BacktestConfig, costs: CostsConfig,
                     cv_cfg: CVConfig, sc_cfg: ScoreConfig,
                     space_n: List[int], space_fast: List[int], space_slow: List[int]) -> Dict:
    best = {"score": -np.inf, "params": None, "report": None}
    rows = []

    for n in space_n:
        for fast in space_fast:
            for slow in space_slow:
                if fast >= slow:
                    continue
                params = {"n": n, "fast": fast, "slow": slow}
                rep = score_params_cv(prices, params, bt_cfg, costs, cv_cfg, sc_cfg)
                rows.append({**params, **rep})
                if rep["score"] > best["score"]:
                    best = {"score": rep["score"], "params": params, "report": rep}

    df = pd.DataFrame(rows).sort_values("score", ascending=False)
    return {"best": best, "table": df}


# =============================================================================
# BENCHMARKS (NO IMPUTATION FOR REGRESSION)
# =============================================================================

def download_benchmark(ticker: str, start: str, end: str) -> pd.Series:
    data = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(data, pd.DataFrame) and "Close" in data.columns:
        s = data["Close"]
    else:
        s = data.squeeze()
    return pd.Series(s).dropna()

def align_returns_no_fill(bench_price: pd.Series, idx: pd.DatetimeIndex) -> pd.Series:
    p = pd.Series(bench_price).reindex(idx)
    return p.pct_change()  # NaNs se resuelven con dropna en la regresión


# =============================================================================
# STRESS EPISODES
# =============================================================================

def evaluate_stress_episodes(prices: pd.DataFrame, params: Dict,
                             bt_cfg: BacktestConfig, costs: CostsConfig,
                             episodes: Dict[str, Tuple[str, str]]) -> pd.DataFrame:
    out = []
    for name, (a, b) in episodes.items():
        sub = prices.loc[a:b]
        if len(sub) < 60:
            continue
        bt = backtest_portfolio(sub, params, bt_cfg, costs)
        m = summarize_performance(bt, bt_cfg)
        out.append({
            "episode": name,
            "start": a,
            "end": b,
            "ann_return": m["ann_return"],
            "sharpe": m["sharpe"],
            "max_dd": m["max_dd"],
            "calmar": m["calmar"],
            "turnover_ann": m["turnover_annualized"],
            "avg_exposure": m["avg_exposure"],
        })
    return pd.DataFrame(out)


# =============================================================================
# MOVING BLOCK BOOTSTRAP (DD + RUIN PROB)
# =============================================================================

def moving_block_bootstrap(series: pd.Series, block: int = 20, n_samples: int = 500, seed: int = 42) -> Dict:
    rng = np.random.default_rng(seed)
    x = series.dropna().values
    T = len(x)
    if T < block * 5:
        return {"dd_p50": np.nan, "dd_p95": np.nan, "ruin_prob_50dd": np.nan}

    dds = []
    for _ in range(n_samples):
        starts = rng.integers(0, T - block, size=int(np.ceil(T / block)))
        sample = np.concatenate([x[s:s + block] for s in starts])[:T]
        equity = (1.0 + sample).cumprod()
        peak = np.maximum.accumulate(equity)
        dd = np.min(equity / peak - 1.0)
        dds.append(dd)

    dds = np.array(dds)
    return {
        "dd_p50": float(np.quantile(dds, 0.50)),
        "dd_p95": float(np.quantile(dds, 0.05)),  # peor 5%
        "ruin_prob_50dd": float(np.mean(dds < -0.5)) * 100.0
    }


# =============================================================================
# REPORTING + PLOTS
# =============================================================================

def report_top_params(cv_table: pd.DataFrame, top: int = 20):
    cols = ["n", "fast", "slow", "score", "avg_sharpe", "std_sharpe", "worst_dd", "avg_calmar", "splits"]
    view = cv_table[cols].head(top).copy()
    print("\nTOP PARAMS (robust score):")
    print(view.to_string(index=False))


def plot_equity_and_drawdown(bt: Dict, benchmarks: Dict[str, pd.Series], bt_cfg: BacktestConfig, title: str):
    eq = bt["equity"]
    dd = (eq / eq.cummax() - 1.0)

    plt.figure()
    plt.plot(eq.index, eq.values, label="Sentinel (net)")
    for name, bret in benchmarks.items():
        bret_aligned = bret.reindex(eq.index).dropna()
        if len(bret_aligned) == 0:
            continue
        beq = bt_cfg.capital_initial * (1.0 + bret_aligned).cumprod()
        plt.plot(beq.index, beq.values, label=name)
    plt.title(f"Equity Curve — {title}")
    plt.legend()
    plt.xlabel("Date")
    plt.ylabel("Equity")
    plt.show()

    plt.figure()
    plt.plot(dd.index, dd.values)
    plt.title(f"Drawdown — {title}")
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.show()


def plot_roll_metrics(bt: Dict, window: int = 126, title: str = ""):
    r = bt["returns_net"].replace([np.inf, -np.inf], np.nan).dropna()
    if len(r) < window * 2:
        print(f"[WARN] Pocos datos para rolling metrics ({window}d).")
        return

    roll_std = r.rolling(window).std()
    roll_sh = (r.rolling(window).mean() / roll_std).replace([np.inf, -npinf], np.nan) * np.sqrt(252)
    roll_vol = roll_std * np.sqrt(252)

    plt.figure()
    plt.plot(roll_sh.index, roll_sh.values)
    plt.title(f"Rolling Sharpe ({window}d) — {title}")
    plt.xlabel("Date")
    plt.ylabel("Sharpe")
    plt.show()

    plt.figure()
    plt.plot(roll_vol.index, roll_vol.values)
    plt.title(f"Rolling Volatility ({window}d) — {title}")
    plt.xlabel("Date")
    plt.ylabel("Ann. Vol")
    plt.show()


def plot_return_distribution(bt: Dict, title: str = ""):
    r = bt["returns_net"].replace([np.inf, -np.inf], np.nan).dropna()
    plt.figure()
    plt.hist(r.values, bins=60)
    plt.title(f"Return Distribution — {title}")
    plt.xlabel("Daily return")
    plt.ylabel("Frequency")
    plt.show()


def plot_exposure_turnover(bt: Dict, title: str = ""):
    w = bt["weights"]
    exposure = w.abs().sum(axis=1)
    turnover = bt["turnover"]

    plt.figure()
    plt.plot(exposure.index, exposure.values)
    plt.title(f"Exposure — {title}")
    plt.xlabel("Date")
    plt.ylabel("Gross exposure")
    plt.show()

    plt.figure()
    plt.plot(turnover.index, turnover.values)
    plt.title(f"Turnover (daily) — {title}")
    plt.xlabel("Date")
    plt.ylabel("Turnover")
    plt.show()


def print_perf_summary(perf: Dict, ab_spy: Dict, ab_qqq: Dict, boot: Dict, title: str):
    print("\n" + "-" * 100)
    print(f"RESUMEN — {title}")
    print("-" * 100)
    print(f"Ann.Return:      {perf['ann_return']*100:10.2f}%")
    print(f"Sharpe (net):    {perf['sharpe']:10.3f} | Sortino: {perf['sortino']:10.3f}")
    print(f"MaxDD:           {perf['max_dd']*100:10.2f}% | Calmar:  {perf['calmar']:10.3f}")
    print(f"Turnover (ann):  {perf['turnover_annualized']:10.2f} | Avg Exposure: {perf['avg_exposure']*100:7.1f}%")
    print(f"CVaR 5%:         {perf['cvar_5']*100:10.3f}% | CVaR 1%: {perf['cvar_1']*100:10.3f}%")
    print("-" * 100)
    print("ALFA/BETA (OLS + Newey–West)")
    print(f"Vs SPY: alpha_ann={ab_spy['alpha_annual']*100:8.2f}% | beta={ab_spy['beta']:7.3f} | t(alpha)={ab_spy['t_alpha']:7.2f}")
    print(f"Vs QQQ: alpha_ann={ab_qqq['alpha_annual']*100:8.2f}% | beta={ab_qqq['beta']:7.3f} | t(alpha)={ab_qqq['t_alpha']:7.2f}")
    print("-" * 100)
    print("BOOTSTRAP (moving blocks)")
    print(f"DD p50:         {boot['dd_p50']*100:10.2f}%")
    print(f"DD p95 (worst): {boot['dd_p95']*100:10.2f}%")
    print(f"P(DD<-50%):     {boot['ruin_prob_50dd']:10.2f}%")
    print("-" * 100)


# =============================================================================
# RUN: SINGLE SCENARIO
# =============================================================================

def run_sentinel81(
    escenario_name: str,
    tickers: List[str],
    bt_cfg: BacktestConfig,
    costs: CostsConfig,
    cv_cfg: CVConfig,
    sc_cfg: ScoreConfig,
    space_n: List[int],
    space_fast: List[int],
    space_slow: List[int],
    benchmark_mkt: str = "SPY",
    benchmark_alt: str = "QQQ",
    make_plots: bool = True,
    top_params_to_print: int = 15
) -> Dict:
    print("=" * 110)
    print(f"SENTINEL 8.1 — ESCENARIO: {escenario_name} | {len(tickers)} tickers")
    print(f"Periodo: {bt_cfg.start} -> {bt_cfg.end} | Rebalance: {bt_cfg.rebalance_mode} | DD cap: {sc_cfg.dd_cap:.0%}")
    print("=" * 110)

    prices = download_prices(tickers, bt_cfg.start, bt_cfg.end)
    if prices.empty or prices.shape[1] < 2:
        raise RuntimeError("Datos insuficientes para backtest/validación. Revise tickers/fechas.")

    # Param search (robust score)
    print("\n[1] Grid Search (score robusto con Purged WFCV + embargo)...")
    search = grid_search_kama(prices, bt_cfg, costs, cv_cfg, sc_cfg, space_n, space_fast, space_slow)
    best = search["best"]
    table = search["table"]
    best_params = best["params"]

    print(f"\nMejores params: {best_params} | score={best['score']:.3f} "
          f"| avgSharpe={best['report']['avg_sharpe']:.3f} | stdSharpe={best['report']['std_sharpe']:.3f} "
          f"| worstDD(CV)={best['report']['worst_dd']:.2%} | splits={best['report']['splits']}")

    report_top_params(table, top=top_params_to_print)

    # Backtest full-period con params elegidos
    print("\n[2] Backtest full-period con params elegidos...")
    bt = backtest_portfolio(prices, best_params, bt_cfg, costs)
    perf = summarize_performance(bt, bt_cfg)

    # Benchmarks + alpha/beta NW
    print("\n[3] Benchmarks + Alfa/Beta (sin imputación; OLS + Newey–West)...")
    bm_spy = align_returns_no_fill(download_benchmark(benchmark_mkt, bt_cfg.start, bt_cfg.end), prices.index)
    bm_qqq = align_returns_no_fill(download_benchmark(benchmark_alt, bt_cfg.start, bt_cfg.end), prices.index)

    ab_spy = alpha_beta_nw(bt["returns_net"], bm_spy, bt_cfg.rf_annual, bt_cfg.trading_days, lags=5)
    ab_qqq = alpha_beta_nw(bt["returns_net"], bm_qqq, bt_cfg.rf_annual, bt_cfg.trading_days, lags=5)

    # Stress episodes
    print("\n[4] Stress histórico por episodios reales...")
    stress = evaluate_stress_episodes(prices, best_params, bt_cfg, costs, STRESS_EPISODES)

    # Bootstrap blocks
    print("\n[5] Block bootstrap (DD y prob. ruina)...")
    boot = moving_block_bootstrap(bt["returns_net"], block=20, n_samples=500, seed=42)

    # Summary print
    print_perf_summary(perf, ab_spy, ab_qqq, boot, title=f"{escenario_name} (FULL PERIOD)")

    if not stress.empty:
        print("\nSTRESS EPISODES:")
        df = stress.copy()
        for c in ["ann_return", "max_dd", "avg_exposure"]:
            df[c] = df[c].astype(float)
        df["ann_return"] = (df["ann_return"] * 100).round(2)
        df["max_dd"] = (df["max_dd"] * 100).round(2)
        df["avg_exposure"] = (df["avg_exposure"] * 100).round(1)
        print(df.to_string(index=False))

    # Plots
    if make_plots:
        title = f"{escenario_name} | params={best_params}"
        benchmarks_for_plot = {
            benchmark_mkt: bm_spy,
            benchmark_alt: bm_qqq,
        }
        plot_equity_and_drawdown(bt, benchmarks_for_plot, bt_cfg, title=title)
        plot_roll_metrics(bt, window=126, title=title)
        plot_return_distribution(bt, title=title)
        plot_exposure_turnover(bt, title=title)

    return {
        "scenario": escenario_name,
        "tickers": list(prices.columns),
        "best_params": best_params,
        "cv_table": table,
        "backtest": bt,
        "performance": perf,
        "alpha_beta_spy": ab_spy,
        "alpha_beta_qqq": ab_qqq,
        "stress": stress,
        "bootstrap": boot,
        "best_cv_report": best["report"],
    }


# =============================================================================
# RUN: MULTICONTEXT
# =============================================================================

def run_multicontext_sentinel81(make_plots_each: bool = False) -> List[Dict]:
    bt_cfg = BacktestConfig(
        capital_initial=100000,
        start="2006-01-01",
        end="2026-02-20",
        burn_in=200,
        rf_annual=0.0,
        rebalance_mode="event",   # recomendado
        rebalance_freq="W-FRI"
    )

    costs = CostsConfig(
        commission=0.001,
        swap_daily=0.0003,
        slippage=0.0002,
        apply_swap=True,
        apply_slippage=True
    )

    cv_cfg = CVConfig(
        train_years=8,
        test_years=1,
        purge_days=10,
        embargo_days=5
    )

    # DD cap acordado
    sc_cfg = ScoreConfig(
        dd_cap=-0.35,
        lambda_dd=2.0,
        lambda_var=0.5,
        min_splits=5
    )

    # Grid (coarse). Luego puede hacer refine alrededor del top.
    space_n = [20, 30, 40, 50, 60, 80, 100]
    space_fast = [2, 3, 5]
    space_slow = [30, 40, 50, 60, 80, 100]

    results = []
    for name, tickers in ESCENARIOS.items():
        try:
            res = run_sentinel81(
                escenario_name=name,
                tickers=tickers,
                bt_cfg=bt_cfg,
                costs=costs,
                cv_cfg=cv_cfg,
                sc_cfg=sc_cfg,
                space_n=space_n,
                space_fast=space_fast,
                space_slow=space_slow,
                benchmark_mkt="SPY",
                benchmark_alt="QQQ",
                make_plots=make_plots_each,
                top_params_to_print=10
            )
            results.append(res)
        except Exception as e:
            print(f"[ERROR] Escenario {name}: {e}")

    # Resumen comparativo multicontexto
    if results:
        rows = []
        for r in results:
            p = r["performance"]
            rows.append({
                "Scenario": r["scenario"],
                "Params": str(r["best_params"]),
                "CV_avgSharpe": round(r["best_cv_report"]["avg_sharpe"], 3),
                "CV_stdSharpe": round(r["best_cv_report"]["std_sharpe"], 3),
                "CV_worstDD%": round(r["best_cv_report"]["worst_dd"] * 100, 2),
                "AnnRet%": round(p["ann_return"] * 100, 2),
                "Sharpe": round(p["sharpe"], 3),
                "MaxDD%": round(p["max_dd"] * 100, 2),
                "Calmar": round(p["calmar"], 3),
                "TurnoverAnn": round(p["turnover_annualized"], 2),
                "AlphaSPY%": round(r["alpha_beta_spy"]["alpha_annual"] * 100, 2) if r["alpha_beta_spy"]["alpha_annual"] == r["alpha_beta_spy"]["alpha_annual"] else np.nan,
                "tAlphaSPY": round(r["alpha_beta_spy"]["t_alpha"], 2) if r["alpha_beta_spy"]["t_alpha"] == r["alpha_beta_spy"]["t_alpha"] else np.nan,
                "RuinProb%": round(r["bootstrap"]["ruin_prob_50dd"], 2) if r["bootstrap"]["ruin_prob_50dd"] == r["bootstrap"]["ruin_prob_50dd"] else np.nan,
            })
        df = pd.DataFrame(rows).sort_values(["Sharpe", "CV_avgSharpe"], ascending=False)

        print("\n" + "=" * 110)
        print("RESUMEN MULTICONTEXTO — SENTINEL 8.1")
        print("=" * 110)
        print(df.to_string(index=False))

        # opcional: export
        df.to_csv("sentinel81_multicontext_summary.csv", index=False)
        print("\n[INFO] Exportado: sentinel81_multicontext_summary.csv")

    return results


# =============================================================================
# ENTRYPOINT
# =============================================================================

if __name__ == "__main__":
    # Si quiere plots para cada escenario, ponga True (se abrirán muchas ventanas).
    results = run_multicontext_sentinel81(make_plots_each=False)

    # Si quiere analizar a profundidad un escenario y ver plots:
    # 1) descomente y ajuste el nombre:
    # single = run_sentinel81(
    #     escenario_name="tech_original",
    #     tickers=ESCENARIOS["tech_original"],
    #     bt_cfg=BacktestConfig(rebalance_mode="event"),
    #     costs=CostsConfig(),
    #     cv_cfg=CVConfig(),
    #     sc_cfg=ScoreConfig(dd_cap=-0.35),
    #     space_n=[20,30,40,50,60,80,100],
    #     space_fast=[2,3,5],
    #     space_slow=[30,40,50,60,80,100],
    #     make_plots=True
    # )

SENTINEL 8.1 — ESCENARIO: tech_original | 12 tickers
Periodo: 2006-01-01 -> 2026-02-20 | Rebalance: event | DD cap: -35%

[1] Grid Search (score robusto con Purged WFCV + embargo)...

Mejores params: {'n': 50, 'fast': 5, 'slow': 40} | score=0.343 | avgSharpe=0.752 | stdSharpe=0.819 | worstDD(CV)=-18.67% | splits=12

TOP PARAMS (robust score):
 n  fast  slow    score  avg_sharpe  std_sharpe  worst_dd  avg_calmar  splits
50     5    40 0.342773    0.752302    0.819059 -0.186677    1.333615      12
50     5   100 0.325777    0.749437    0.847320 -0.216415    1.363190      12
50     5    50 0.312555    0.740740    0.856369 -0.219169    1.348540      12
50     5    60 0.311252    0.739164    0.855824 -0.228339    1.345989      12
60     3   100 0.307935    0.710727    0.805583 -0.165946    1.263562      12
40     5    80 0.287791    0.723531    0.871480 -0.224889    1.363386      12
40     5   100 0.286122    0.718759    0.865274 -0.229263    1.338090      12
60     5    80 0.278216    0.71